In [79]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from tqdm import tqdm

In [80]:
train = pd.read_csv(r"C:/Users/fatem/Downloads/ChiFraud_train.csv", sep='\t', encoding='utf-8')
test = pd.read_csv(r"C:/Users/fatem/Downloads/ChiFraud_t2023.csv", sep='\t', encoding='utf-8')

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train columns: ['Label_id', 'Text']
Test columns: ['Label_id', 'Text']
Train shape: (192267, 2)
Test shape: (115553, 2)


In [81]:
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-chinese",
    num_labels=11
)

print("Model configured for 11 classes")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model configured for 11 classes


In [82]:
class FraudDataset(Dataset):
    def __init__(self, df, text_column="Text", label_column="Label_id"):
        self.texts = df[text_column].astype(str).tolist()
        self.labels = df[label_column].astype(int).tolist()
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoded = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [83]:
small_train = train.sample(frac=0.01, random_state=42)
train_ds = FraudDataset(small_train)
test_ds = FraudDataset(test)

print(f"Training samples: {len(train_ds)} (1% of original)")
print(f"Test samples: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)  # Use larger batch size
test_loader = DataLoader(test_ds, batch_size=32)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Training samples: 1923 (1% of original)
Test samples: 115553
Training batches: 61
Test batches: 3612


In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

Using device: cpu


In [85]:
print("Starting training...")
model.train()


small_train = train.sample(frac=0.01, random_state=42)
small_train_ds = FraudDataset(small_train)
small_train_loader = DataLoader(small_train_ds, batch_size=32, shuffle=True)

print(f"Training on {len(small_train_ds)} samples only")

for epoch in range(1):
    total_loss = 0
    loop = tqdm(small_train_loader, desc=f"Epoch {epoch}")
    
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        optimizer.step()
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(small_train_loader)
    print(f"Epoch {epoch} - Average Loss: {avg_loss:.4f}")

print("Quick training completed!")

Starting training...
Training on 1923 samples only


Epoch 0: 100%|█████████████████████████████████████████████████████████████| 61/61 [27:24<00:00, 26.96s/it, loss=0.201]

Epoch 0 - Average Loss: 0.6957
Quick training completed!


In [87]:
print("Starting QUICK evaluation...")
model.eval()
predictions = []
true_labels = []

small_test = test.sample(frac=0.01, random_state=42)
small_test_ds = FraudDataset(small_test)
small_test_loader = DataLoader(small_test_ds, batch_size=32)

print(f"Evaluating on {len(small_test_ds)} samples only")

with torch.no_grad():
    for batch in tqdm(small_test_loader, desc="Quick Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)
print(f"Model accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(true_labels, predictions))

Starting QUICK evaluation...
Evaluating on 1156 samples only


Quick Evaluating: 100%|████████████████████████████████████████████████████████████████| 37/37 [04:01<00:00,  6.53s/it]
C:\Users\fatem\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Model accuracy: 0.8503

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94       945
           1       0.00      0.00      0.00        45
           2       0.46      0.54      0.50        81
           3       0.00      0.00      0.00         8
           4       0.00      0.00      0.00         6
           5       0.00      0.00      0.00        34
           6       0.00      0.00      0.00         7
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00        12
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00        11

    accuracy                           0.85      1156
   macro avg       0.12      0.14      0.13      1156
weighted avg       0.76      0.85      0.80      1156



C:\Users\fatem\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\fatem\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [88]:
def predict_text(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
    
    return pred

test_text = "尊敬的用户，您的银行卡异常，请立即点击此链接验证。"
prediction = predict_text(test_text)
print(f"Text: {test_text}")
print(f"Predicted class: {prediction}")

Text: 尊敬的用户，您的银行卡异常，请立即点击此链接验证。
Predicted class: 0


In [89]:
test_texts = [
    "尊敬的用户，您的银行卡异常，请立即点击此链接验证。",
    "Hello, how are you today?",
    "您的账户需要验证，请提供个人信息",
    "This is a normal message",
    "紧急通知：您的账户已被冻结"
]

for text in test_texts:
    prediction = predict_text(text)
    print(f"Text: {text}")
    print(f"Predicted class: {prediction}\n")

Text: 尊敬的用户，您的银行卡异常，请立即点击此链接验证。
Predicted class: 0

Text: Hello, how are you today?
Predicted class: 0

Text: 您的账户需要验证，请提供个人信息
Predicted class: 0

Text: This is a normal message
Predicted class: 0

Text: 紧急通知：您的账户已被冻结
Predicted class: 0



In [90]:
# Test the same examples again
test_texts = [
    "尊敬的用户，您的银行卡异常，请立即点击此链接验证。",  # Likely fraud
    "Hello, how are you today?",  # Normal English
    "您的账户需要验证，请提供个人信息",  # Likely fraud  
    "This is a normal message",  # Normal English
    "紧急通知：您的账户已被冻结"  # Likely fraud
]

print("Testing predictions after proper training:")
for text in test_texts:
    prediction = predict_text(text)
    print(f"Text: {text}")
    print(f"Predicted class: {prediction}\n")

Testing predictions after proper training:
Text: 尊敬的用户，您的银行卡异常，请立即点击此链接验证。
Predicted class: 0

Text: Hello, how are you today?
Predicted class: 0

Text: 您的账户需要验证，请提供个人信息
Predicted class: 0

Text: This is a normal message
Predicted class: 0

Text: 紧急通知：您的账户已被冻结
Predicted class: 0

